# CHEM 269 — Tier-1 Conformer Pipeline (Colab Pro · A100 · High RAM)

Generates dual-environment (aqueous / membrane) RDKit conformer descriptors for all 7,298 CycPeptMPDB molecules.

**Runtime:** Select **A100 GPU + High RAM** in Runtime → Change runtime type before running.

> Note: ETKDGv3 / MMFF94s conformer generation is CPU-bound. The A100 runtime is chosen for its **12 CPU cores + 83 GB RAM**, not for GPU computation.

---

### What to upload to Google Drive before running

```
MyDrive/
└── chem269_tier1/
    └── pampa_curated.csv          ← REQUIRED: copy from data/pampa_curated.csv
    └── results/                   ← create this empty folder (Drive will auto-create)
```

**To resume a partial run:** also upload `results/conformer_descriptors_checkpoint.csv` (from local or a prior Colab session) into `MyDrive/chem269_tier1/results/` before running Cell 5.

---

### Output files written to `MyDrive/chem269_tier1/results/`

| File | Written | Description |
|---|---|---|
| `conformer_descriptors_checkpoint.csv` | Every 200 mols | Incremental save — safe to interrupt |
| `run_log.csv` | Every 200 mols | Per-molecule timing + error log |
| `conformer_descriptors_raw.csv` | On completion | Final descriptor table (all molecules) |
| `run_summary.txt` | On completion | Stats: counts, failures, descriptor distributions |

---

### Cell order

| Cell | Action | Notes |
|---|---|---|
| 1 | Install RDKit | ~1–2 min |
| 2 | Mount Drive | Sign in when prompted |
| 3 | Configuration | Edit paths here if needed |
| 4 | Load functions | No edits needed |
| 5 | Load data + resume check | Shows how many mols remain |
| 6 | **Run pipeline** | Main loop — leave running |
| 7 | Results summary | Run after loop finishes |
| 8 | Download results | Optional — files already on Drive |

In [ ]:
# ━━ CELL 1: Install RDKit ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Pure pip install — no condacolab needed. Takes ~1-2 min.
!pip install -q rdkit tqdm

import multiprocessing, rdkit, tqdm
print(f'RDKit  : {rdkit.__version__}')
print(f'tqdm   : {tqdm.__version__}')
print(f'CPUs   : {multiprocessing.cpu_count()}')

# Confirm high-RAM runtime
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'RAM    : {ram_gb:.0f} GB')
if ram_gb < 50:
    print('WARNING: RAM < 50 GB — are you on High RAM runtime?')
else:
    print('High RAM runtime confirmed.')

RDKit  : 2025.09.6
tqdm   : 4.67.3
CPUs   : 12
RAM    : 179 GB
High RAM runtime confirmed.


In [ ]:
# ━━ CELL 2: Mount Google Drive ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('Drive mounted at /content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


In [ ]:
# ━━ CELL 3: Configuration ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Edit these if your Drive folder is named differently.
import multiprocessing, os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/chem269_tier1')
INPUT_CSV    = DRIVE_ROOT / 'pampa_curated.csv'
RESULTS_DIR  = DRIVE_ROOT / 'results'

# Conformer settings
N_CONFS          = 20    # 50 for screening; 200 for final production run
N_CPUS           = multiprocessing.cpu_count()   # uses all cores (12 on A100)
CHUNKSIZE        = 8     # imap_unordered chunksize — reduces IPC overhead
CHECKPOINT_EVERY = 200   # save to Drive every N completed molecules

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f'Input CSV not found: {INPUT_CSV}\n'
        f'Upload pampa_curated.csv to MyDrive/chem269_tier1/'
    )

print(f'Input CSV  : {INPUT_CSV}')
print(f'Results dir: {RESULTS_DIR}')
print(f'n_confs={N_CONFS}  n_cpus={N_CPUS}  chunksize={CHUNKSIZE}  checkpoint_every={CHECKPOINT_EVERY}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ━━ CELL 4: Processing functions ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Identical logic to scripts/conformer_engine.py — do not edit.
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors3D, rdFreeSASA
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore')

HB_DONOR_SMARTS    = Chem.MolFromSmarts('[N,O;!H0]')
HB_ACCEPTOR_SMARTS = Chem.MolFromSmarts('[N,O]')
HB_DIST_CUTOFF     = 3.0
HB_ANGLE_CUTOFF    = 120.0

_BONDI = {
    'H': 1.20, 'C': 1.70, 'N': 1.55, 'O': 1.52,
    'S': 1.80, 'P': 1.80, 'F': 1.47, 'Cl': 1.75,
    'Br': 1.85, 'I': 1.98,
}
_POLAR_ELEMENTS = {'N', 'O', 'S', 'P'}


def _polar_sasa(mol, conf_id):
    try:
        radii = []
        for atom in mol.GetAtoms():
            sym = atom.GetSymbol()
            radii.append(_BONDI.get(sym, 1.50))
            if sym in _POLAR_ELEMENTS:
                atom.SetIntProp('SASAClass', 0)
                atom.SetProp('SASAClassName', 'Polar')
            else:
                atom.SetIntProp('SASAClass', 1)
                atom.SetProp('SASAClassName', 'APolar')
        query = rdFreeSASA.MakeFreeSasaPolarAtomQuery()
        return round(rdFreeSASA.CalcSASA(mol, radii, confIdx=conf_id, query=query), 4)
    except Exception:
        return np.nan


def _intramolecular_hbonds(mol, conf_id):
    try:
        pos      = mol.GetConformer(conf_id).GetPositions()
        donors   = [i for m in mol.GetSubstructMatches(HB_DONOR_SMARTS)    for i in m]
        acceptors= [i for m in mol.GetSubstructMatches(HB_ACCEPTOR_SMARTS) for i in m]
        count    = 0
        for d_idx in donors:
            for h_atom in mol.GetAtomWithIdx(d_idx).GetNeighbors():
                if h_atom.GetAtomicNum() != 1:
                    continue
                h_pos = pos[h_atom.GetIdx()]
                d_pos = pos[d_idx]
                for a_idx in acceptors:
                    if a_idx == d_idx:
                        continue
                    try:
                        if len(Chem.GetShortestPath(mol, d_idx, a_idx)) < 6:
                            continue
                    except Exception:
                        continue
                    if np.linalg.norm(h_pos - pos[a_idx]) > HB_DIST_CUTOFF:
                        continue
                    vec_hd = d_pos - h_pos
                    vec_ha = pos[a_idx] - h_pos
                    cos    = np.dot(vec_hd, vec_ha) / (
                        np.linalg.norm(vec_hd) * np.linalg.norm(vec_ha) + 1e-9
                    )
                    if np.degrees(np.arccos(np.clip(cos, -1, 1))) >= HB_ANGLE_CUTOFF:
                        count += 1
        return count
    except Exception:
        return np.nan


def _shape_descriptors(mol, conf_id):
    try:
        return {
            'Rg':             Descriptors3D.RadiusOfGyration(mol, confId=conf_id),
            'NPR1':           Descriptors3D.NPR1(mol, confId=conf_id),
            'NPR2':           Descriptors3D.NPR2(mol, confId=conf_id),
            'Asphericity':    Descriptors3D.Asphericity(mol, confId=conf_id),
            'Eccentricity':   Descriptors3D.Eccentricity(mol, confId=conf_id),
            'SpherocityIndex':Descriptors3D.SpherocityIndex(mol, confId=conf_id),
            'PBF':            Descriptors3D.PBF(mol, confId=conf_id),
        }
    except Exception:
        return {k: np.nan for k in
                ['Rg','NPR1','NPR2','Asphericity','Eccentricity','SpherocityIndex','PBF']}


def process_molecule(args):
    """Embed → minimize → select aq/mem conformers → return Δ descriptors."""
    import time
    mol_id, smiles, n_confs = args
    t0 = time.time()

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {'ID': mol_id, 'error': 'invalid_smiles', 'wall_s': 0.0}

    try:
        mol = rdMolStandardize.FragmentParent(mol)
        mol = rdMolStandardize.Uncharger(canonicalOrder=True).uncharge(mol)
        Chem.SanitizeMol(mol)
    except Exception:
        pass

    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed            = 42
    params.maxIterations         = 2000
    params.numThreads            = 1
    params.useSmallRingTorsions  = True
    params.useMacrocycleTorsions = True
    params.pruneRmsThresh        = 0.5

    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if len(conf_ids) == 0:
        params.pruneRmsThresh = 1.0
        conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=params)
    if len(conf_ids) == 0:
        return {'ID': mol_id, 'error': 'embed_failed', 'wall_s': round(time.time()-t0, 2)}

    mmff = AllChem.MMFFOptimizeMoleculeConfs(
        mol, mmffVariant='MMFF94s', maxIters=2000, numThreads=1
    )
    converged = [conf_ids[i] for i,(nc,_) in enumerate(mmff) if nc == 0] or list(conf_ids)

    per_conf = []
    for cid in converged:
        per_conf.append({
            'conf_id':    cid,
            'mmff_energy':next((e for i,(_,e) in enumerate(mmff) if conf_ids[i]==cid), np.nan),
            'psa3d':      _polar_sasa(mol, cid),
            'hb_count':   _intramolecular_hbonds(mol, cid),
            **_shape_descriptors(mol, cid),
        })

    if not per_conf:
        return {'ID': mol_id, 'error': 'no_descriptors', 'wall_s': round(time.time()-t0, 2)}

    df_c = pd.DataFrame(per_conf).dropna(subset=['psa3d'])
    if df_c.empty:
        return {'ID': mol_id, 'error': 'psa_failed', 'wall_s': round(time.time()-t0, 2)}

    aq  = df_c.loc[df_c['psa3d'].idxmax()]
    mem = df_c.loc[df_c['psa3d'].idxmin()]

    return {
        'ID':                   mol_id,
        'n_confs_generated':    len(conf_ids),
        'n_confs_used':         len(df_c),
        # Aqueous conformer (max PSA — polar groups exposed)
        'aq_psa3d':             float(aq['psa3d']),
        'aq_hb_count':          float(aq['hb_count']),
        'aq_Rg':                float(aq['Rg']),
        'aq_NPR1':              float(aq['NPR1']),
        'aq_NPR2':              float(aq['NPR2']),
        'aq_Asphericity':       float(aq['Asphericity']),
        'aq_SpherocityIndex':   float(aq['SpherocityIndex']),
        # Membrane conformer (min PSA — polar groups buried)
        'mem_psa3d':            float(mem['psa3d']),
        'mem_hb_count':         float(mem['hb_count']),
        'mem_Rg':               float(mem['Rg']),
        'mem_NPR1':             float(mem['NPR1']),
        'mem_NPR2':             float(mem['NPR2']),
        'mem_Asphericity':      float(mem['Asphericity']),
        'mem_SpherocityIndex':  float(mem['SpherocityIndex']),
        # Delta features (chameleonic potential)
        'delta_psa3d':          float(aq['psa3d']       - mem['psa3d']),
        'delta_hb':             float(mem['hb_count']   - aq['hb_count']),
        'delta_Rg':             float(aq['Rg']          - mem['Rg']),
        'delta_NPR1':           float(aq['NPR1']        - mem['NPR1']),
        'delta_NPR2':           float(aq['NPR2']        - mem['NPR2']),
        'delta_Asphericity':    float(aq['Asphericity'] - mem['Asphericity']),
        # PSA spread across all conformers
        'psa3d_spread':         float(df_c['psa3d'].max() - df_c['psa3d'].min()),
        'psa3d_std':            float(df_c['psa3d'].std()),
        'hb_spread':            float(df_c['hb_count'].max() - df_c['hb_count'].min()),
        'error':                None,
        'wall_s':               round(time.time() - t0, 2),
    }


print('Processing functions loaded.')

Processing functions loaded.


In [ ]:
# ━━ CELL 5: Load data + resume check ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import pandas as pd
from pathlib import Path

CKPT_PATH  = RESULTS_DIR / 'conformer_descriptors_checkpoint.csv'
FINAL_PATH = RESULTS_DIR / 'conformer_descriptors_raw.csv'
LOG_PATH   = RESULTS_DIR / 'run_log.csv'

# Load full dataset
df = pd.read_csv(INPUT_CSV, low_memory=False)
smiles_col = 'SMILES_canonical' if 'SMILES_canonical' in df.columns else 'SMILES'
df = df[['ID', smiles_col, 'PAMPA']].dropna(subset=[smiles_col]).copy()
print(f'Dataset       : {len(df):,} molecules')
print(f'PAMPA range   : {df["PAMPA"].min():.2f} → {df["PAMPA"].max():.2f}')

# Check for existing output — treat completed raw CSV as a resumable checkpoint
# so we can continue from 1,500 → 7,298 without re-running finished molecules.
done_ids = set()
ckpt_df  = pd.DataFrame()

if CKPT_PATH.exists():
    ckpt_df  = pd.read_csv(CKPT_PATH, low_memory=False)
    done_ids = set(ckpt_df['ID'].astype(str))
    print(f'\nResuming from checkpoint: {len(done_ids):,} already done.')
elif FINAL_PATH.exists():
    # Previous run finished (checkpoint was deleted) but dataset was smaller.
    # Load the final CSV as a pseudo-checkpoint so those molecules are skipped.
    ckpt_df  = pd.read_csv(FINAL_PATH, low_memory=False)
    done_ids = set(ckpt_df['ID'].astype(str))
    print(f'\nLoaded prior final output as checkpoint: {len(done_ids):,} already done.')
    print(f'(Will extend to full {len(df):,}-molecule dataset.)')
else:
    print('\nNo checkpoint — starting fresh.')

tasks = [
    (row['ID'], row[smiles_col], N_CONFS)
    for _, row in df.iterrows()
    if str(row['ID']) not in done_ids
]
print(f'Remaining     : {len(tasks):,} molecules to process')
print(f'Workers       : {N_CPUS} CPUs × {N_CONFS} conformers each')

In [ ]:
# ━━ CELL 6: Run pipeline ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Checkpoints to Drive every CHECKPOINT_EVERY molecules.
# Interrupt anytime — re-run from Cell 5 to resume.

import multiprocessing as mp
import time
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

CKPT_PATH    = RESULTS_DIR / 'conformer_descriptors_checkpoint.csv'
FINAL_PATH   = RESULTS_DIR / 'conformer_descriptors_raw.csv'
LOG_PATH     = RESULTS_DIR / 'run_log.csv'
SUMMARY_PATH = RESULTS_DIR / 'run_summary.txt'

if not tasks:
    print('No remaining tasks — all molecules already processed. Run Cell 7 for summary.')
else:
    n_total = len(tasks)
    print(f'Starting: {n_total:,} molecules · {N_CPUS} workers · {N_CONFS} confs each')
    print(f'Checkpointing every {CHECKPOINT_EVERY} molecules to Drive.')
    print(f'Results dir: {RESULTS_DIR}')
    print('─' * 60)

    results  = []
    log_rows = []
    t_start  = time.time()

    with mp.Pool(N_CPUS) as pool:
        for res in tqdm(
            pool.imap_unordered(process_molecule, tasks, chunksize=CHUNKSIZE),
            total=n_total,
            desc='Conformers',
            unit='mol',
        ):
            results.append(res)

            log_rows.append({
                'ID':     res.get('ID'),
                'error':  res.get('error'),
                'wall_s': res.get('wall_s', None),
            })

            if len(results) % CHECKPOINT_EVERY == 0:
                batch_df = pd.DataFrame(results)
                combined = pd.concat([ckpt_df, batch_df], ignore_index=True)
                combined.to_csv(CKPT_PATH, index=False)

                log_df = pd.DataFrame(log_rows)
                log_df.to_csv(LOG_PATH, mode='a', header=not LOG_PATH.exists(), index=False)
                log_rows = []

                elapsed = time.time() - t_start
                avg_s   = elapsed / len(results)
                eta_h   = avg_s * (n_total - len(results)) / 3600
                n_total_done = len(done_ids) + len(results)
                print(
                    f'  [ckpt] {n_total_done:>5}/{len(df):>5} done · '
                    f'{avg_s:.1f}s/mol · ETA {eta_h:.1f}h'
                )

    if log_rows:
        pd.DataFrame(log_rows).to_csv(LOG_PATH, mode='a', header=not LOG_PATH.exists(), index=False)

    all_results = pd.concat(
        [ckpt_df, pd.DataFrame(results)], ignore_index=True
    ) if not ckpt_df.empty else pd.DataFrame(results)

    all_results.to_csv(FINAL_PATH, index=False)
    if CKPT_PATH.exists():
        CKPT_PATH.unlink()

    elapsed_total = time.time() - t_start
    ok = all_results[all_results['error'].isna()]
    n_failed = all_results['error'].notna().sum()

    summary_lines = [
        'CHEM 269 Tier-1 Run Summary',
        '=' * 40,
        f'Total molecules  : {len(all_results):,}',
        f'Successful       : {len(ok):,}',
        f'Failed           : {n_failed:,}',
        f'Wall time        : {elapsed_total/3600:.2f} h',
        f'Avg time / mol   : {elapsed_total/max(len(results),1):.1f} s',
        f'n_confs          : {N_CONFS}',
        f'n_cpus           : {N_CPUS}',
        '',
    ]

    if n_failed:
        summary_lines += ['Failure breakdown:', all_results['error'].value_counts().to_string(), '']

    if len(ok) > 0:
        summary_lines.append('Descriptor statistics (successful molecules):')
        for col in ['aq_psa3d','mem_psa3d','delta_psa3d','delta_hb','psa3d_spread','psa3d_std']:
            if col in ok.columns:
                v = ok[col].dropna()
                summary_lines.append(
                    f'  {col:<22}: mean={v.mean():>7.2f}  std={v.std():>6.2f}  '
                    f'min={v.min():>7.2f}  max={v.max():>7.2f}'
                )

    summary_text = '\n'.join(summary_lines)
    SUMMARY_PATH.write_text(summary_text)
    print('\n' + summary_text)
    print(f'\nOutputs written to {RESULTS_DIR}:')
    for f in sorted(RESULTS_DIR.iterdir()):
        print(f'  {f.name:<45} {f.stat().st_size/1e6:.1f} MB')

In [ ]:
# ━━ CELL 7: Results summary (run anytime after Cell 6 starts) ━━━━━━━━━━━━━━━━
import pandas as pd
from pathlib import Path

CKPT_PATH  = RESULTS_DIR / 'conformer_descriptors_checkpoint.csv'
FINAL_PATH = RESULTS_DIR / 'conformer_descriptors_raw.csv'
LOG_PATH   = RESULTS_DIR / 'run_log.csv'

p = FINAL_PATH if FINAL_PATH.exists() else CKPT_PATH if CKPT_PATH.exists() else None

if p is None:
    print('No results yet — run Cell 6 first.')
else:
    res = pd.read_csv(p)
    ok  = res[res['error'].isna()]
    print(f'File     : {p.name}  ({p.stat().st_size/1e6:.1f} MB)')
    print(f'Rows     : {len(res):,}  (successful: {len(ok):,}  failed: {res["error"].notna().sum():,})')

    if res['error'].notna().any():
        print('\nFailure breakdown:')
        print(res['error'].value_counts().to_string())

    if len(ok) > 0:
        print('\nDescriptor statistics:')
        cols = ['aq_psa3d','mem_psa3d','delta_psa3d','delta_hb','psa3d_spread','psa3d_std']
        print(ok[[c for c in cols if c in ok.columns]].describe().round(2).to_string())

    if LOG_PATH.exists():
        log = pd.read_csv(LOG_PATH)
        print(f'\nRun log  : {len(log):,} entries')
        if 'wall_s' in log.columns:
            wt = log['wall_s'].dropna()
            print(f'Wall time: mean={wt.mean():.1f}s  max={wt.max():.0f}s  total={wt.sum()/3600:.1f}h')

File     : conformer_descriptors_raw.csv  (0.5 MB)
Rows     : 1,500  (successful: 1,500  failed: 0)

Descriptor statistics:
       aq_psa3d  mem_psa3d  delta_psa3d  delta_hb  psa3d_spread  psa3d_std
count   1500.00    1500.00      1500.00   1500.00       1500.00    1499.00
mean     164.91      93.68        71.23      1.40         71.23      19.00
std       40.92      25.43        23.59      1.71         23.59       6.08
min       78.82      35.64         0.00     -4.00          0.00       6.38
25%      125.37      72.32        51.38      0.00         51.38      13.89
50%      170.05      95.10        69.32      1.00         69.32      18.84
75%      196.89     112.02        87.23      2.00         87.23      23.42
max      313.95     313.95       163.40      8.00        163.40      39.16

Run log  : 1,500 entries
Wall time: mean=123.7s  max=632s  total=51.5h


In [ ]:
# ━━ CELL 8: Download results to local machine ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Files are already on Drive — this cell downloads them to your computer.
from google.colab import files
from pathlib import Path

FINAL_PATH   = RESULTS_DIR / 'conformer_descriptors_raw.csv'
CKPT_PATH    = RESULTS_DIR / 'conformer_descriptors_checkpoint.csv'
LOG_PATH     = RESULTS_DIR / 'run_log.csv'
SUMMARY_PATH = RESULTS_DIR / 'run_summary.txt'

to_download = [FINAL_PATH, CKPT_PATH, LOG_PATH, SUMMARY_PATH]

downloaded = 0
for p in to_download:
    if p.exists():
        print(f'Downloading {p.name} ({p.stat().st_size/1e6:.1f} MB)...')
        files.download(str(p))
        downloaded += 1

if downloaded == 0:
    print('No output files found yet — run Cell 6 first.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>